# 이미지를 읽고 설명하기 - LLM을 이용

In [2]:
!pip install google-genai python-dotenv

In [12]:
from PIL import Image
from google import genai
import os, io
import base64 # 이미지를 문자열로 변경
from dotenv import load_dotenv
from google.genai import types

load_dotenv() # 경로가 다르면 적어줘야함

api_key = os.getenv("GOOGLE_API_KEY")
# Gemini Client 객체 생성
client = genai.Client(api_key=api_key)
model = 'gemini-2.5-flash'

# 프롬프트 작성
system_prompt = "당신은 그림을 보고 내용에 대한 설명을 잘하는 전문가입니다." # 모델의 역할 주기(페르소나)
user_prompt = "다음 이미지를 보고 무슨 내용인지 작성하시오" # 실제 사용자 요청 내용

# 이미지 열기
image_path = 'pic.jpeg'
with open(image_path, 'rb') as f:
  image_data = f.read()
  image = Image.open(io.BytesIO(image_data)).convert("RGB")
  print(image.size) # (658, 465)

# 이미지를 base64 문자열로 변경 - OPENAI api가 이걸 원함
base64_image = base64.b64encode(image_data).decode('utf-8')
# print(base64_image)

(658, 465)


In [15]:
# 멀티 모달(텍스트 + 이미지) 질의 실행 - 단일 실행형 스크립트
response = client.models.generate_content(
    model=model,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt
    ),
    contents=[
        {
            "role": "user",
            "parts": [
                {
                    "text": user_prompt
                },
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": base64_image
                    }
                }
            ]
        }
    ]
)
# print(response)
print("이미지 설명 :")

if response.text:
    print(response.text)
else:
    print("응답이 비었어요")

이미지 설명 :
이 이미지는 맑고 푸른 하늘 아래 광활하게 펼쳐진 자연 풍경을 담고 있습니다.

1.  **하늘**: 화면의 대부분을 차지하는 하늘은 깊고 선명한 파란색으로, 한 점 티 없이 맑은 날씨를 보여줍니다. 군데군데 뭉게뭉게 피어있는 하얀 구름들은 그림에 입체감과 청량감을 더하며 시원한 느낌을 선사합니다. 넓게 트인 하늘은 보는 이에게 시원함과 개방감을 안겨줍니다.

2.  **초원/잔디밭**: 하늘 아래로는 끝없이 펼쳐진 푸른 초원 또는 잔디밭이 완만한 경사를 이루며 시원하게 뻗어 있습니다. 신선하고 생기 넘치는 초록빛 잔디가 시선을 멀리 이끌며, 건강하고 활기찬 자연의 모습을 보여줍니다.

3.  **한 그루의 나무**: 푸른 잔디밭 한가운데에는 잎이 무성한 한 그루의 나무가 우뚝 서 있습니다. 이 나무는 이미지의 시각적인 초점 역할을 하며, 넓은 공간 속에서 존재감을 드러내 고요하고 평화로운 분위기를 강조합니다. 나무의 푸른 잎사귀는 싱그러움을 더합니다.

4.  **지평선의 숲**: 잔디밭 너머 지평선으로는 짙은 녹색의 숲이 길게 이어져 있어 전체적인 풍경에 깊이감을 더하고, 시야의 끝을 부드럽게 마무리해 줍니다.

**전반적인 분위기**:
이 이미지는 드넓은 자연 속에서 느껴지는 평화로움과 고요함, 그리고 맑고 깨끗한 공기가 느껴지는 듯한 상쾌함을 전달합니다. 답답함 없이 탁 트인 시야는 보는 이에게 안정감과 해방감을 안겨주며, 휴식과 명상을 위한 이상적인 공간처럼 느껴집니다.


In [17]:
# 클래스 기반 구조화
class MyMultiModal:
    def __init__(self, client, model, system_prompt="", user_prompt=""):
        self.client = client
        self.model = model
        self.system_prompt = system_prompt
        self.user_prompt = user_prompt

    # 이미지 경로를 받아 스트리밍 응답 생성
    def streamMethod(self, image_path):
        with open(image_path, "rb") as f:
            image_bytes = f.read()

        base64_image = base64.b64encode(image_bytes).decode("utf-8")

        response = self.client.models.generate_content_stream(
            model=self.model,
            config=types.GenerateContentConfig(
                system_instruction=self.system_prompt
            ),
            contents=[
                {
                    "role": "user",
                    "parts": [
                        {
                            "text": self.user_prompt
                        },
                        {
                            "inline_data": {
                                "mime_type": "image/jpeg",
                                "data": base64_image
                            }
                        }
                    ]
                }
            ]
        )

        return response

# 스트리밍 응답 출력 함수
def stream_responseFunc(response):
    for chunk in response:
        if chunk.text:
            print(chunk.text, end="", flush=True)

model = "gemini-2.5-flash"
system_prompt = "당신은 시인입니다. 당신의 의무는 주어진 이미지를 가지고 시를 작성하는 것입니다."
user_prompt = "다음 이미지를 보고 시를 작성하시오"

multimodal_gemini = MyMultiModal(
    client=client,
    model=model,
    system_prompt=system_prompt,
    user_prompt=user_prompt
)

IMAGE_PATH = "pic.jpeg"

response = multimodal_gemini.streamMethod(IMAGE_PATH)
stream_responseFunc(response)

**들판의 노래, 하늘의 꿈**

깊고 푸른 하늘, 끝없이 펼쳐진
솜털 같은 구름 양떼, 느리게 흘러가네
푸른 비단 깔린 듯, 광활한 들판 위
바람결에 흔들리는 초록 물결 넘실거려

그 넓은 품에 홀로, 굳건히 서 있는 나무
대지의 심장을 박고 하늘을 향해 뻗은
수천 개의 잎새들, 햇살 받아 반짝이며
고요한 생명의 노래를 바람에 실어 보내네

바람이 나뭇가지 스쳐 갈 때면
초록 잎사귀들 푸른 속삭임 주고받고
따스한 햇살 아래, 넓은 그늘 드리우며
지친 영혼에게 쉼을 허락하네

세상 모든 번뇌 잠재우는 듯
단순한 풍경 속에 담긴 깊은 평화
홀로 서 있지만 홀로가 아닌 듯
하늘과 땅, 구름과 들판과 하나 되어

저 나무처럼 굳건히, 저 들판처럼 너그럽게
숨 쉬고 싶은 고요, 바라보고 싶은 영원
하늘과 땅 사이에 펼쳐진
가장 아름다운 한 폭의 시.

---
### Open Ai Api 버전

In [ ]:
# 멀티 모달(텍스트 + 이미지) 질의 실행 - Openai api용
# 단일 실행형 스크립트
response = client.responses.create(
    model = model.
    input = [
        {
            # 첫번째 전달할 메세지 : system역할 메세지
            "role":"system", # 모델 역할, 태도, 규칙, 답변형식 등 지정
            "content":[{"type":"input_text","text":system_prompt}]
        },
        {
            # 두번째 전달할 메세지 : user역할 메세지
            "role":"user", # 사용자의 실제 질문, 요청, 이미지, 텍스트 전달...
            "content":[{"type":"input_text","text":user_prompt}]
        },
        {
           "type":"input_image",
           "image_url":f"dtat:image/jpeg;base64,{base64_image}"
        }

    ]
)
# print(response)
print("이미지 설명 :")
if response.output_text:
  print(response.output_text)
else:
  print("응답이 비었어요")

In [ ]:
# 클래스 기반 구조화
class MyMultiModal:
  def __init__(self, client, model, system_prompt="", user_prompt=""):
     self.client = client
     self.model = model
     self.system_prompt = system_prompt
     self.user_prompt = user_prompt

  # 이미지 경로를 받아 스트리밍 응답을 생성
  def stremMethod(self, image_path):
    with open(image_path, 'rb') as f:
      image_bytes = f.read()

    base64_image = base64.b64encode(image_bytes).decode('utf-8')
    self.response = self.client.responses.create(
          model = self.model.
          input = [
              {
                  # 첫번째 전달할 메세지 : system역할 메세지
                  "role":"system", # 모델 역할, 태도, 규칙, 답변형식 등 지정
                  "content":[{"type":"input_text","text":self.system_prompt}]
              },
              {
                  # 두번째 전달할 메세지 : user역할 메세지
                  "role":"user", # 사용자의 실제 질문, 요청, 이미지, 텍스트 전달...
                  "content":[{"type":"input_text","text":self.user_prompt}]
              },
              {
                "type":"input_image",
                "image_url":f"dtat:image/jpeg;base64,{base64_image}"
              }

          ],
          stream=True # 스트리밍 응답으로 생성되는 텍스트가 한 줄씩 실시간 출력됨
      )
    return response


# 스트리밍 응답 출력 함수
def stream_responseFunc(response):
  for event in response:  # 스트리밍으로 전달되는 이벤트를 하나씩 수행

    # delta : 스트리밍중에 조금씩 도착하는 '응답 조각' - 응답을 한번에 받아서 조금씩 출력하겠다.
    if event.type == 'response.output_text.delta':
      print(event.delta, end="", flush=True) # flush=True -> 버퍼 대기 없이 바로 출력

llm = 'gpt-4o-mini'
system_prompt = "당신은 시인입니다. 당신의 의무는 주어진 이미지를 가지고 시를 작성하는 것입니다." # 모델의 역할 주기(페르소나)
user_prompt = "다음 이미지를 보고 시를 작성하시오" # 실제 사용자 요청 내용

multimodal_gpt = MyMultiModal(
    client=client,
    model=llm,
    system_prompt=system_prompt,
    user_prompt=user_prompt
)

IMAGE_PATH = 'pic.jpeg'
response = multimodal_gpt.stremMethod(IMAGE_PATH)
stream_responseFunc(response)